In [30]:
#Imports 
import pandas as pd
import re
import numpy as np
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Paths
BRONZE_PATH = "../data/bronze/"
SILVER_PATH = "../data/silver/"



In [31]:
# Load all bronze files
print("Loading bronze files")
cert_bronze = pd.read_csv(os.path.join(BRONZE_PATH, "certifications_bronze.csv"))
clubs_bronze = pd.read_csv(os.path.join(BRONZE_PATH, "clubs_bronze.csv"))
results_bronze = pd.read_csv(os.path.join(BRONZE_PATH, "results_bronze.csv"))

print("\n" + "="*60)
print("BRONZE FILE SHAPES (Before cleaning)")
print("="*60)
print(f"Certifications: {cert_bronze.shape} rows, {cert_bronze.shape[1]} columns")
print(f"Clubs: {clubs_bronze.shape} rows, {clubs_bronze.shape[1]} columns")
print(f"Results: {results_bronze.shape} rows, {results_bronze.shape[1]} columns")

Loading bronze files

BRONZE FILE SHAPES (Before cleaning)
Certifications: (21001, 12) rows, 12 columns
Clubs: (436, 19) rows, 19 columns
Results: (112375, 14) rows, 14 columns


In [32]:
print("="*60)
print("SILVER: Cleaning Certifications")
print("="*60)

df = cert_bronze.copy()
print(f"Starting shape: {df.shape}")

# Step 1: Standardize Person type
df['person_type_clean'] = df['Person type'].astype(str).str.lower().str.strip()
df['person_type_clean'] = df['person_type_clean'].replace('nan', None)
print(f"  Person type unique values: {df['person_type_clean'].unique()}")

# Step 2: Standardize Gender
df['gender_clean'] = df['Gender'].astype(str).str.upper()
df['gender_clean'] = df['gender_clean'].replace('NAN', None)
print(f"  Gender unique values: {df['gender_clean'].unique()}")

# Step 3: Convert boolean columns to 0/1
bool_cols = [
    'Mental Handicap (SOB has this certificate)',
    'Parents Consent (SOB has this certificate)',
    'HAP (SOB has this certificate)',
    'Unified Partner (SOB has this certificate)'
]

for col in bool_cols:
    df[col] = df[col].fillna(0).astype(int)
    df[col] = df[col].astype(str).str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0}).fillna(0).astype(int)
print(f"  Boolean columns converted to 0/1")

# Step 4: Calculate Age from DOB
df['dob_date'] = pd.to_datetime(df['DOB'], errors='coerce')
current_year = 2024
df['calculated_age'] = current_year - df['dob_date'].dt.year

# Use existing Age column where available
df['calculated_age'] = df['calculated_age'].fillna(df['Age'])

# Calculate median age from valid entries (age between 5 and 100)
valid_ages = df[(df['calculated_age'] >= 5) & (df['calculated_age'] <= 100)]['calculated_age']
median_age = valid_ages.median() if len(valid_ages) > 0 else 30
print(f"  Median age from valid entries: {median_age:.1f} years")

# Fill missing ages with median
before_fill = df['calculated_age'].isnull().sum()
df['calculated_age'] = df['calculated_age'].fillna(median_age)
print(f"  Filled {before_fill} missing ages with median")

# Step 5: Flag missing DOB
df['dob_missing'] = df['DOB'].isnull().astype(int)
print(f"  DOB missing: {df['dob_missing'].sum()} rows ({df['dob_missing'].sum()/len(df)*100:.1f}%)")

# Remove rows with missing Code
before_remove = len(df)
df = df[df['Code'].notna() & (df['Code'].astype(str).str.strip() != '')]
df = df[df['Code'].astype(str).str.len() > 0]
after_remove = len(df)
print(f"  Removed {before_remove - after_remove} rows with missing Code (athlete_id)")

# Step 7: For remaining rows, fill other nulls where possible
df['gender_clean'] = df['gender_clean'].fillna('U')
df['person_type_clean'] = df['person_type_clean'].fillna('unknown')

print(f"\nFinal shape: {df.shape}")

# Final null check
print(f"Null counts after cleaning:")
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print(remaining_nulls if len(remaining_nulls) > 0 else "  No null values remaining")

# Save to silver
# Step 8: Select only the columns we want for Silver (drop raw/original columns)
columns_to_keep = [
    # Cleaned identifiers
    'Code',
    'person_type_clean',
    'gender_clean',
    
    # Cleaned demographic data
    'calculated_age',
    'dob_missing',
    
    # Boolean flags (already cleaned)
    'Mental Handicap (SOB has this certificate)',
    'Parents Consent (SOB has this certificate)',
    'HAP (SOB has this certificate)',
    'Unified Partner (SOB has this certificate)',
    
    # Metadata (keep for traceability)
    'bronze_timestamp'
]

# Select only these columns
df_clean = df[columns_to_keep].copy()

# Rename for clarity
df_clean = df_clean.rename(columns={
    'Mental Handicap (SOB has this certificate)': 'mental_handicap_flag',
    'Parents Consent (SOB has this certificate)': 'parents_consent_flag',
    'HAP (SOB has this certificate)': 'hap_flag',
    'Unified Partner (SOB has this certificate)': 'unified_partner_flag'
})

print(f"  Silver columns: {df_clean.columns.tolist()}")
print(f"  Saving shape: {df_clean.shape}")

# Save to silver
output_path = os.path.join(SILVER_PATH, "certifications_silver.csv")
df_clean.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

SILVER: Cleaning Certifications
Starting shape: (21001, 12)
  Person type unique values: <ArrowStringArray>
[        'athlete',           'coach', 'unified partner',               nan,
       'volunteer',             'vip',           'staff',         'medical',
      'head coach',         'manager',        'security',   'family member',
           'a-hod',           'media',        'as-staff']
Length: 15, dtype: str
  Gender unique values: <ArrowStringArray>
['M', 'F', nan, 'U']
Length: 4, dtype: str
  Boolean columns converted to 0/1
  Median age from valid entries: 37.0 years
  Filled 780 missing ages with median
  DOB missing: 1990 rows (9.5%)
  Removed 780 rows with missing Code (athlete_id)

Final shape: (20221, 17)
Null counts after cleaning:
DOB         1210
dob_date    1210
dtype: int64
  Silver columns: ['Code', 'person_type_clean', 'gender_clean', 'calculated_age', 'dob_missing', 'mental_handicap_flag', 'parents_consent_flag', 'hap_flag', 'unified_partner_flag', 'bronze_times

In [33]:
print("="*60)
print("SILVER: Cleaning Clubs")
print("="*60)

df = clubs_bronze.copy()
print(f"Starting shape: {df.shape}")

# Step 1: Rename columns
df = df.rename(columns={
    'Group number': 'club_id',
    'Name': 'club_name',
    'Province': 'region'
})
print(f"  Renamed columns: club_id, club_name, region")

# Step 2: Fix Country - all should be Belgium
if 'Country' in df.columns:
    before = df['Country'].isnull().sum()
    df['Country'] = df['Country'].fillna('Belguq')
    # Fix any variations
    df['Country'] = df['Country'].apply(lambda x: 'Belgique' if pd.notna(x) and str(x).lower() in ['belgium', 'bel', 'be', 'belgië', 'belgique'] else x)
    print(f"  Country: filled {before} nulls and standardized to 'Belgique'")

# Step 3: Fix region nulls
if 'region' in df.columns:
    before = df['region'].isnull().sum()
    df['region'] = df['region'].fillna('Unknown')
    print(f"  Region: filled {before} nulls with 'Unknown'")

# Step 4: Fix address fields - fill with empty string
address_cols = ['Address (Street and Number)', 'Zipcode', 'City']
for col in address_cols:
    if col in df.columns:
        before = df[col].isnull().sum()
        df[col] = df[col].fillna('')
        print(f"  {col}: filled {before} nulls with empty string")

# Step 5: Find participation columns
participation_cols = [col for col in df.columns if 'Participation Games' in col]
print(f"  Found {len(participation_cols)} participation year columns")

# Step 6: Convert participation flags to 0/1
for col in participation_cols:
    df[col] = df[col].fillna(0).astype(int)
    df[col] = df[col].astype(str).str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0}).fillna(0).astype(int)
print(f"  Participation flags converted to 0/1")

# Step 7: Create total_participations column
df['total_participations'] = df[participation_cols].sum(axis=1)
print(f"  Created total_participations column (range: {df['total_participations'].min()} to {df['total_participations'].max()})")

print(f"\nFinal shape: {df.shape}")

# Final null check
print(f"Null counts after cleaning:")
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print(remaining_nulls if len(remaining_nulls) > 0 else "  No null values remaining")

# Save to silver
# Step 8: Select only the columns we want for Silver (drop raw/original columns)
columns_to_keep = [
    'club_id',
    'club_name',
    'region',
    'Country',
    'total_participations',
    'bronze_timestamp'
]

# Also include participation flags if needed for Gold
participation_cols = [col for col in df.columns if 'Participation Games' in col]
columns_to_keep.extend(participation_cols)

# Select only these columns
df_clean = df[columns_to_keep].copy()

print(f"  Silver columns: {df_clean.columns.tolist()}")
print(f"  Saving shape: {df_clean.shape}")

# Save to silver
output_path = os.path.join(SILVER_PATH, "clubs_silver.csv")
df_clean.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

SILVER: Cleaning Clubs
Starting shape: (436, 19)
  Renamed columns: club_id, club_name, region
  Country: filled 74 nulls and standardized to 'Belgique'
  Region: filled 6 nulls with 'Unknown'
  Address (Street and Number): filled 2 nulls with empty string
  Zipcode: filled 3 nulls with empty string
  City: filled 0 nulls with empty string
  Found 9 participation year columns
  Participation flags converted to 0/1
  Created total_participations column (range: 0 to 9)

Final shape: (436, 20)
Null counts after cleaning:
  No null values remaining
  Silver columns: ['club_id', 'club_name', 'region', 'Country', 'total_participations', 'bronze_timestamp', 'Participation Games 2015', 'Participation Games 2016', 'Participation Games 2017', 'Participation Games 2018', 'Participation Games 2019', 'Participation Games 2022', 'Participation Games 2023', 'Participation Games 2024', 'Participation Games 2025']
  Saving shape: (436, 15)

Saved to: ../data/silver/clubs_silver.csv


In [34]:
df_test = pd.read_csv(os.path.join(SILVER_PATH, "clubs_silver.csv"))
print("Clubs validation:")
print(f"  Shape: {df_test.shape}")
print(f"  Region unique values: {df_test['region'].nunique()}")
print(f"  Total participations range: {df_test['total_participations'].min()} to {df_test['total_participations'].max()}")
print(f"  Sample: {df_test[['club_id', 'club_name', 'region', 'total_participations']].head(2)}")

Clubs validation:
  Shape: (436, 15)
  Region unique values: 25
  Total participations range: 0 to 9
  Sample:    club_id   club_name      region  total_participations
0      100  LA PILERIE     Hainaut                     9
1      101         BAM  Luxembourg                     3


In [35]:
print("="*60)
print("SILVER: Cleaning Results")
print("="*60)

df = results_bronze.copy()
print(f"Starting shape: {df.shape}")

# Step 1: Standardize Gender
df['gender_clean'] = df['Gender'].map({'Male': 'M', 'Female': 'F'})
print(f"  Gender mapping complete")

# Step 2: Fix Gender nulls
df['gender_clean'] = df['gender_clean'].fillna('U')
print(f"  Gender nulls filled: {df['gender_clean'].isnull().sum()}")

# Step 3: Calculate Age from DOB
df['dob_date'] = pd.to_datetime(df['DOB'], errors='coerce')
current_year = 2024
df['calculated_age'] = current_year - df['dob_date'].dt.year

# Use existing Age column where available
df['calculated_age'] = df['calculated_age'].fillna(df['Age'])

# Calculate median age from valid entries
valid_ages = df[(df['calculated_age'] >= 5) & (df['calculated_age'] <= 100)]['calculated_age']
median_age = valid_ages.median() if len(valid_ages) > 0 else 30
print(f"  Median age from valid entries: {median_age:.1f} years")

# Fill missing ages with median
before_fill = df['calculated_age'].isnull().sum()
df['calculated_age'] = df['calculated_age'].fillna(median_age)
print(f"  Filled {before_fill} missing ages with median")

# Step 4: Extract numeric rank from Place column
def extract_rank(place):
    if pd.isna(place):
        return None
    match = re.search(r'(\d+)', str(place))
    return int(match.group(1)) if match else None

df['rank_numeric'] = df['Place'].apply(extract_rank)

# Fill missing rank with 999
before_rank = df['rank_numeric'].isnull().sum()
df['rank_numeric'] = df['rank_numeric'].fillna(999).astype(int)
print(f"  Filled {before_rank} missing ranks with 999")

# Step 5: Create rank_missing_flag
df['rank_missing_flag'] = (df['rank_numeric'] == 999).astype(int)
print(f"  Rank missing flag: {df['rank_missing_flag'].sum()} rows flagged")

# Step 6: Extract numeric score from Score column
def extract_score(score):
    if pd.isna(score):
        return None
    match = re.search(r'(\d+(?:\.\d+)?)', str(score))
    return float(match.group(1)) if match else None

df['score_numeric'] = df['Score'].apply(extract_score)

# Fill missing score with 0
before_score = df['score_numeric'].isnull().sum()
df['score_numeric'] = df['score_numeric'].fillna(0)
print(f"  Filled {before_score} missing scores with 0")

# Step 7: Create score_missing_flag
df['score_missing_flag'] = (df['score_numeric'] == 0).astype(int)
print(f"  Score missing flag: {df['score_missing_flag'].sum()} rows flagged")

# Step 8: Fill missing Place with 'Unknown'
before_place = df['Place'].isnull().sum()
df['Place'] = df['Place'].fillna('Unknown')
print(f"  Filled {before_place} missing Place values with 'Unknown'")

# Step 9: Fill missing Summary with empty string
df['Summary (all)'] = df['Summary (all)'].fillna('')

# Step 10: Flag disqualifications
df['is_disqualified'] = df['Summary (all)'].str.contains('DQ|Disqualified', case=False, na=False).astype(int)
df['is_disqualified'] |= df['Score'].fillna('').str.contains('DQ|Disqualified', case=False, na=False).astype(int)
dq_count = df['is_disqualified'].sum()
print(f"  Disqualifications flagged: {dq_count} rows ({dq_count/len(df)*100:.1f}%)")

# Step 11: Extract main sport
def extract_sport(sport):
    if pd.isna(sport):
        return None
    return str(sport).split('/')[0].strip()

df['sport_clean'] = df['Sport'].apply(extract_sport)
print(f"  Sport extraction complete")
print(f"  Unique sports: {df['sport_clean'].nunique()}")

# Step 12: Ensure year is integer
df['year'] = df['source_year'].astype(int)
print(f"  Years present: {sorted(df['year'].unique())}")

# Step 13: Remove rows with missing required fields
before = len(df)
df = df[df['Code'].notna() & (df['Code'].astype(str).str.strip() != '')]
df = df[df['sport_clean'].notna() & (df['sport_clean'].astype(str).str.strip() != '')]
df = df[df['Club'].notna() & (df['Club'].astype(str).str.strip() != '')]
after = len(df)
print(f"  Removed {before - after} rows with missing Code, Sport, or Club")

print(f"\nFinal shape: {df.shape}")

# Final null check on critical columns
critical_cols = ['Code', 'sport_clean', 'year', 'gender_clean', 'score_numeric', 'rank_numeric']
print(f"\nNull counts on critical columns:")
for col in critical_cols:
    null_count = df[col].isnull().sum()
    print(f"  {col}: {null_count} nulls")

# Step 14: Select only the columns we want for Silver (drop raw/original columns)
columns_to_keep = [
    # Identifiers
    'Code',
    'Club',
    
    # Cleaned data
    'gender_clean',
    'calculated_age',
    'sport_clean',
    'year',
    
    # Numeric measures
    'score_numeric',
    'rank_numeric',
    
    # Flags
    'is_disqualified',
    'rank_missing_flag',
    'score_missing_flag',
    
    # Keep for debugging (optional)
    'bronze_timestamp'
]

# Select only these columns
df_clean = df[columns_to_keep].copy()

# Rename for clarity
df_clean = df_clean.rename(columns={
    'Code': 'athlete_id',
    'Club': 'club_name',
    'gender_clean': 'gender',
    'calculated_age': 'age',
    'sport_clean': 'sport_name',
    'score_numeric': 'score',
    'rank_numeric': 'rank'
})

print(f"  Silver columns: {df_clean.columns.tolist()}")
print(f"  Saving shape: {df_clean.shape}")

# Save to silver
output_path = os.path.join(SILVER_PATH, "results_silver.csv")
df_clean.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")

SILVER: Cleaning Results
Starting shape: (112375, 14)
  Gender mapping complete
  Gender nulls filled: 0
  Median age from valid entries: 41.0 years
  Filled 79 missing ages with median
  Filled 31290 missing ranks with 999
  Rank missing flag: 31290 rows flagged
  Filled 29132 missing scores with 0
  Score missing flag: 43415 rows flagged
  Filled 27430 missing Place values with 'Unknown'
  Disqualifications flagged: 2241 rows (2.0%)
  Sport extraction complete
  Unique sports: 23
  Years present: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Removed 79 rows with missing Code, Sport, or Club

Final shape: (112296, 24)

Null counts on critical columns:
  Code: 0 nulls
  sport_clean: 0 nulls
  year: 0 nulls
  gender_clean: 0 nulls
  score_numeric: 0 nulls
  rank_numeric: 0 nulls
  Silver columns: ['athlete_id', 'club_name', 'gender', 'age', 'sport_name', 'year', 'score', 'rank', 'is_disq